# UFCEK5-30-2 - Big data and generative models
# TB1 Final Assessment (100 marks)

For this part of the assessment, you will need Python, PySpark and TinyDB. We strongly recommend using Google Colab. You may use UWE PCs as well, as we have already covered these topics in both environments. However, we cannot guarantee that automatic updates on UWE PCs will keep PySpark and TinyDB versions aligned.

### Notice 1
Please DO NOT change (add/remove) cells in this notebook. Only write your codes and your comments/reflections in the specified cells. Changing the layout of the notebook/cells, will affect your mark. As stated in the assessment brief, the total number of words in Markdown cells in this Jupyter Notebook needs to be within the specified limit. Please note that this notebook already holds 1467 words in its Markdown cells. When counting words, ignore this amount. You are provided `Markdown_word_count.ipynb` tool to count the number of words.

### Notice 2
Where requested, add at least 1 test case to verify correctness of your code. Add test cases ONLY in the provided cells.

### Notice 3
To achieve higher marks (optional), you would create a repository (such as in Github) for this assessment and commit gradual progress there. Share the link of the repo in the next cell. If your repo is private, share it with the email of all lecturers:

`Ahsan.Kazmi@uwe.ac.uk`

`Alireza.Bolhari@uwe.ac.uk`

`Ramya.Anandan@uwe.ac.uk`




==================================================

## Please write your full name and student ID here:

### Full name: Graham Howard
### Student ID: 24047396
### Repo URL: https://gitlab.uwe.ac.uk/g7-howard/24047396_bigdata_generativemodels/-/tree/main

==================================================

## *Datasets Description*

For this assessment, you are going to work with these 3 datasets:
1) JSON and Audio: a .jsonl dataset with 12 audio files (.wav)
2) Images: a dataset of 60 images
3) Parquet: a dataset of stock market prices

You can download these datasets from BlackBoard.

A brief description about these datasets:

- 1: this is a conversational speech dataset for English ASR (Automatic Speech Recognition). The .json file holds metadata of the original audio files. We have randomly chosen 12 audio files (.wav) for this assessment.

Link to the original dataset: https://huggingface.co/datasets/apptek-com/apptek_callcenter_dialogues

- 2: This is a dataset about self driving cars. We have selected only 60 of the images.

Link to the original dataset: https://public.roboflow.com/object-detection/self-driving-car

- 3: This is a dataset about stock market prices, which is updated daily. Two .parquet files are available. You will only need `stock_prices.parquet`. The other file (`stock_profile.parquet`) is metadata and you may wish to take a look at that one for extra information about each stock symbol.

Link to the original dataset: https://huggingface.co/datasets/defeatbeta/yahoo-finance-data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Uploading datasets to Colab

You may upload `.zip` version of the datasets to Colab and then unzip them using Python code. For instance, to unzip `stock_market.zip`:

In [ ]:
import zipfile

with zipfile.ZipFile('/content/drive/MyDrive/stock_market.zip', 'r') as zip_ref:
    zip_ref.extractall('stock_market')

## Database (TinyDB) Setup

Please ensure you have TinyDB installed. If you need to install it, use this cell:

In [3]:
# BEGIN SOLUTION
"""
According to Siemens (2021, sect. Installing TinyDB) the TinyDB package is installed into Google Collab (or a notebook) using the command "pip install tinydb". It is further
stated by that the cammand be requires a prefix of "!" or "%" for Google Collab to recognise the "pip" as a command (otherwise Python raises a Sytax Error error) and those
prefixes perform the similar task of installing a package, however, these prefixes engage the packages installation using different methods  (GeeksforGeeks, 2024,
sect. [Using Pip in a Code Cell] and [Using the %pip Magic Command]). As referenced by GEN01, the "!pip" comand method required use of an additional terminal (running in the
background) which could result erronously engaging a different terminal and kernal, whereas "%pip" runs on the notebook's existing termminal and kernel only with no risk
of running in a difference environment. Hence, the "%pip" command method has been used to install TinyDB in the Collab notebook environment.
"""
%pip install tinydb

# END SOLUTION

If you need to `import` anything, please ONLY use this cell to put ALL the `import` statements for this notebook:

In [5]:
# Put ALL the import statements for this notebook in this cell.
from tinydb import TinyDB, Query
import json as js
from pathlib import Path
import pandas as pd


## 1: Create a database
Create a database and name it your student ID: `student_id.json`. For example, if your student ID is `12345678`, your database should be: `12345678.json`. Then store your database in a variable named `db`.

In [6]:
# BEGIN SOLUTION
# The
db = TinyDB("24047396.json")
print('Json file for assignment is successfully loaded')
# END SOLUTION

Json file for assignment is successfully loaded


## 2: Loading call centre dataset

Write a Python code to load the JSONL file (`call_centre_metadata.jsonl`) and insert all of its records into your database. Then print the number of documents in the database.

In [7]:
# BEGIN SOLUTION
# Your code here with comments or explanation
callcentre = Path("/content/call_centre_metadata.jsonl")
Q2assignment = Path("/content/24047396.json")

Q2source = pd.read_json(callcentre, lines=True)
Q2source.to_json(Q2assignment, orient="records") ## load data into the assignment 24047396.json file
Q2target = pd.read_json(Q2assignment, orient="records") ## Read from assignment json database after insertion of data from source
Q2target.head(5)

JSONread_originstate = Q2source #pd.read_json(Q2assignment, orient="records")

# END SOLUTION

In [8]:
# The tests written for Question 2 are repeatable for answering other questions, therefore these tests have been built as functions
# so they are repeatable.

def data_reconciliation_test(source, target):
  source_len, target_len = int(len(source)), int(len(target))

  if source_len == target_len:
    result_rec = ('Data Reconciliation test passed')
  else:
    result_rec = ('Data Reconciliation test FAILED')

  print(f"{result_rec}. Source JSON file has {source_len} records, and Target JSON file has {target_len} records")

  return


def data_compare_test(source, target):#, length):
  s_concat = source.astype(str).agg(''.join, axis=1)
  t_concat = target.astype(str).agg(''.join, axis=1)

  compare_table = pd.DataFrame({'SourceFile': s_concat, 'TargetFile': t_concat})

  error_file, passed, failed, count = [], 0, 0, 0
  for count, (i,j) in enumerate(compare_table.itertuples(index=False, name=None), start=1):
    if i == j:
      passed +=1
    else:
      failed +=1
      error_file.append(f"Target row {count} row does not match Source row {count}")
    count +=1

  if failed == 0:
    error_list = None
  else:
  #failed > 0:
    error_list = (f'Here is the list of the {failed} documents that failed:\n {pd.DataFrame(error_file)}.')

  print(f'Data Comparison tests: Passed = {passed}, Failed = {failed}. {error_list}')

  return

#Q2target_testfile = Path("/content/call_centre_metadata_testQ2_target.jsonl")
#Q2target_test = pd.read_json(target_testfile, lines=True)

data_reconciliation_test(Q2source, Q2target)
data_compare_test(Q2source, Q2target)

Data Reconciliation test passed. Source JSON file has 67 records, and Target JSON file has 67 records
Data Comparison tests: Passed = 67, Failed = 0. None


#### START.
Your analysis / interpretation goes here:



#### END.

## 3: Listing `.wav` files
Write a Python code to read all `.wav` audio files, search in your database (use `file_name` key) and print the ID of the document with the `.wav` file. For example, the output needs to be something like this:


 | Doc ID      | File Name |
| ----------- | ----------- |
| 1      | en_GB_Agriculture_1582366.wav       |
| 3   | en_GB_Aviation_1582210.wav        |

In [11]:
# BEGIN SOLUTION
# Your code here with comments or explanation
Q3source = Q2target.copy()

Q3staging = Q3source.copy()
Q3staging['doc_id'] = range(1, len(Q3staging) + 1)
Q3staging['file_name_new'] = Q3staging['file_name'].str[6:]
Q3staging.insert(0, 'doc_id', Q3staging.pop('doc_id'))
Q3staging.insert(1, 'file_name_new', Q3staging.pop('file_name_new'))
Q3staging = Q3staging.drop(columns=['file_name'])
Q3staging = Q3staging.rename(columns={'doc_id': 'Doc ID', 'file_name_new': 'File Name', 'duration': 'Duration'
                                    , 'domain': 'Domain', 'accent': 'Accent', 'segments': 'Segments'})

Q3staging.to_json(Q2assignment, orient='records')

Q3target = pd.read_json(Q2assignment, orient='records')
Q3target.head(3)

,Doc ID,File Name,Duration,Domain,Accent,Segments
0,1,en_GB_Agriculture_1582366.wav,477.520,agriculture,en-GB,"[{'start': 12.94, 'end': 21.14, 'role': 'custo..."
1,2,en_GB_Agriculture_1582397.wav,670.037,agriculture,en-GB,"[{'start': 13.24, 'end': 17.54, 'role': 'agent..."
2,3,en_GB_Aviation_1582210.wav,509.420,aviation,en-GB,"[{'start': 11.82, 'end': 22.14, 'role': 'agent..."


## 4: Storing file paths
You are asked to store these `.wav` audio files in a path like `files/audio/file_name` in your NoSQL database. So, for the first audio file, the path will be `files/audio/en_GB_Agriculture_1582366.wav`. Now, add a new key to your respective document ID named `file_path` and record the path of all audio files there.

In [12]:
# BEGIN SOLUTION
# couldn't find the procedure so used GenAI to find best code structure. Note: upon review of json file, JSON has replaced the forward
# slash "/" with "\/" in the raw data because JSON used this it represent aforward slash. When printed or exported in dataframe
# pandas replaces visual back to single forwards slash.

Q4source = Q3target.copy()
print('\nQ4source\n', Q4source.head(2).to_markdown())

Q4staging = Q4source.copy()
Q4staging['File Path'] = 'files/audio/' + Q4staging['File Name']
Q4staging.insert(1, 'File Path', Q4staging.pop('File Path'))
print('\nQ4staging\n', Q4staging.head(2).to_markdown())

Q4staging.to_json(Q2assignment, orient='records')

Q4target = pd.read_json(Q2assignment, orient="records")
print('\nQ4target\n', Q4target.head(2).to_markdown())

# END SOLUTION


Q4source
 |    |   Doc ID | File Name                     |   Duration | Domain      | Accent   | Segments                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [13]:
# Initial attempts to test the source & target data proived troublesome because Pandas JSON replaces single slashes "/" with mirrored slashes "\/"
# hence prevents a comparison due taregt data having "\/" but source data "/".
# This has been solved by importing the source data into the Q4target_db TinyDB json file with updated filepath that can now be effectively compared
# with taget data.

#Q4source_data = JSONread_originstate.copy()

Q4source_data =  Q3target.copy()
Q4source_data['TESTDATAfile_path'] = 'files/audio/' + Q4source_data['File Name']
Q4source_data.insert(1, 'TESTDATAfile_path', Q4source_data.pop('TESTDATAfile_path'))

Q4testfilepath = Path("/content/call_centre_metadata_testQ4_source.json")
Q4negtest = pd.read_json(Q4testfilepath, orient="records")

Q4expectedresult = Q4source_data
Q4actualresult = Q4target.copy()

print("********\nTest Case 1: Every document should be updated with a File path key (testing the contents of the key is different test)")
data_reconciliation_test(Q4expectedresult['TESTDATAfile_path'].dropna(), Q4actualresult['File Path'].dropna())

print("********\nTest Case 2: The File Path key if suffixed with file/audio/<File Name>")
filepath = pd.DataFrame()
filepath[['ExpectedFP']] = Q4actualresult[['File Path']]
filepath['ActualFP'] = 'files/audio/' + Q4actualresult['File Name']

filepath['pathresult'] = (filepath['ExpectedFP'] == filepath['ActualFP']).map({True: 'Passed', False: 'Failed'})
passed = (filepath['pathresult'] == 'Passed').sum()
failed = (filepath['pathresult'] == 'Failed').sum()

print(f"Path reconciliation: Passed = {passed}, Failed = {failed}")
if failed > 0:
  print(f'Here is the list of the {failed} documents that failed:\n {(filepath['pathresult'] == 'Passed')}.')

print("********\nTest Case 3: The count of documents should be unaffected by the filepath update")
data_reconciliation_test(Q4expectedresult, Q4actualresult)

print("********\nTest Case 4: The keys of documents brought-forward from source should be unaffected by the File Path update, i.e nothign has changed")
data_compare_test(Q4expectedresult, Q4actualresult)


#print('\nQ4source_data\n', Q4source_data.head(5).to_markdown())
#print('\nQ4negtest\n', Q4negtest.head(15).to_markdown())
#print('\nQ4actualresult\n', Q4actualresult.head(5).to_markdown())





********
Test Case 1: Every document should be updated with a File path key (testing the contents of the key is different test)
Data Reconciliation test passed. Source JSON file has 67 records, and Target JSON file has 67 records
********
Test Case 2: The File Path key if suffixed with file/audio/<File Name>
Path reconciliation: Passed = 67, Failed = 0
********
Test Case 3: The count of documents should be unaffected by the filepath update
Data Reconciliation test passed. Source JSON file has 67 records, and Target JSON file has 67 records
********
Test Case 4: The keys of documents brought-forward from source should be unaffected by the File Path update, i.e nothign has changed
Data Comparison tests: Passed = 67, Failed = 0. None



Why recording the `file_path` in your database? Why not the file itself? Write your answer in the Markdown cell below.

#### START.
Your analysis / interpretation goes here:



#### END.

## 5: Self driving car dataset

There are 60 images in a folder named `self_drive_car` next to this Jupyter Notebook. Insert them into your database, with OBJECTIDs starting from 4000 and counting thereafter. For example, the first image will have: `OBJECTID: 4000` and the second one will have `OBJECTID: 4001`. Use the following structure to insert these images as documents into your database. Note that:

- in the `file_name`, you need to store EXACTLY the same name as the name of the file, WITHOUT its extension.
- in the `data`, you need to store the converted version of the image (base64).


Note: Please make sure that stored base64 images are decodable to the original images.

In [ ]:
# DO NOT WRITE ANY CODE IN THIS CELL. PLEASE USE THE NEXT CELL FOR YOUR ANSWER.

'''
"OBJECTID": 4000
"file_type": "image/jpg"
"object_purpose": "self_driving_car"
"file_name": "The name of the file"
"data": "base64 string"
'''

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.




#### START.
Your analysis / interpretation goes here:



#### END.

## PySpark - Environment Setup

From this part, you will need PySpark as well. Use the following cell to install PySpark.
Run the cell below once per Colab session. The `-q` flag keeps the output quiet.

Note 1: Remember that you will need to work with your NoSQL database. So, do not close your connection with the TinyDB.

Note 2: After question 14, there is a single Markdown cell for your analysis and interpretation of findings for PySpark.

In [ ]:
!pip install pyspark -q

### Start a Spark Session
Every PySpark script needs a `SparkSession`. This is the entry point to using Spark. Create a session named as your student ID. For instance, if your student ID is 12345678, the session must be named `12345678`.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

## 6: Load the stock market dataset

Write a code to load the `stock_market.parquet` dataset into a Spark DataFrame named `df`.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.




## 7: Loading & understanding the data

Work through each question in order (later questions build on columns created in earlier ones).

Load `stock_prices.parquet`, print its schema, and report:
- the total number of rows
- the number of distinct stock symbols

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation




# END SOLUTION

Considering this dataset, what other exploratory data analysis would you do to have a better understanding of the dataset? Write them as bullet points in the Markdown cell below.

#### START.
Your analysis / interpretation goes here:



#### END.

## 8: Date conversion

Check the data type of `report_date`. If it is not already a proper date type, convert it to `DateType`. Then create two new columns, `year` and `month`, derived from `report_date`.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

## 9: Feature engineering - 2 new columns

Create two new columns:
- `daily_range` = `high` − `low`
- `daily_return_pct` = `(close − open) / open × 100`

Note: For the second one, you need to handle where `open` price is zero. Something like: `F.when(F.col("open") != 0, ...`

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.




## 10: Largest daily range

Which single trading day (across all symbols) had the largest `daily_range`? Show the symbol, date, and value.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation




# END SOLUTION

## 11: Aggregation

For each symbol, compute: average close price, average volume, maximum high, minimum low, and number of trading days. Sort by average volume descending and show the top 10 symbols.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

## 12: Volatility

Define a symbol's *volatility* as the standard deviation of its `daily_return_pct`. Which 5 symbols are the most volatile?

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

## 13: Closing price moving average

For each symbol, calculate a 7-day (trailing, inclusive) moving average of `close`, ordered by `report_date`. Add the new value to a column named `ma_7_close`. Show the result for symbol `"A"` only, for its first 10 trading days.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

## 14: All-time high closing price

For each symbol, find the date on which it reached its all-time-high closing price. Show one row per symbol: `symbol`, `report_date`, `close`.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

#### START.
Your analysis / interpretation for PySpark tasks (8-14) go here:



#### END.

## 15: Inserting into database - stock summary

Insert the per-symbol summary table from question 11 into your existing TinyDB database, in a NEW table called `stock_summary` (one record per symbol).

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.




#### START.
Your analysis / interpretation goes here:



#### END.

## 16: Inserting into database - volatile

Insert your top-5 most volatile symbols from 12 into a new table called `volatility_ranking`. Each record should include a `rank` field from 1 (most volatile) to 5.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation



# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.




#### START.
Your analysis / interpretation goes here:



#### END.

## 17: Your Final Reflection

In less than 1500 words, reflect on how you approached this assessment, what was new to you and how you found solutions. Moreover, relate your experience on this assessment to Big Data's four Vs (volume, variety, velocity, and veracity). Choose 2 of them and give one concrete example from your own implementation for those.

#### START
Your reflection here:




#### END